In [115]:
!pip install earthengine-api geemap -q

import ee
import geemap

ee.Authenticate()
ee.Initialize(project='ee-kirschenworks')

In [116]:
gaul = ee.FeatureCollection("FAO/GAUL/2015/level1")

ubon = gaul.filter(
    ee.Filter.And(
        ee.Filter.eq("ADM0_NAME", "Thailand"),
        ee.Filter.eq("ADM1_NAME", "Ubon Ratchathani")
    )
)

geometry = ubon.geometry()

In [117]:
s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
        .filterBounds(geometry) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))

def maskS2(image):
    scl = image.select('SCL')
    mask = scl.neq(3) \
        .And(scl.neq(8)) \
        .And(scl.neq(9)) \
        .And(scl.neq(10)) \
        .And(scl.neq(11))
    return image.updateMask(mask)

s2 = s2.map(maskS2)

In [118]:
dry = s2.filterDate('2023-01-01', '2023-04-30') \
        .median() \
        .clip(geometry)

wet = s2.filterDate('2023-07-01', '2023-11-30') \
        .median() \
        .clip(geometry)

In [119]:
def add_indices(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndwi = image.normalizedDifference(['B3', 'B8']).rename('NDWI')
    return image.addBands([ndvi, ndwi])

dry = add_indices(dry)
wet = add_indices(wet)

In [120]:
dem = ee.Image("USGS/SRTMGL1_003").clip(geometry)

In [121]:
Map = geemap.Map()
Map.centerObject(geometry, 8)

ndvi_params = {'min':0, 'max':1, 'palette':['brown','yellow','green']}
ndwi_params = {'min':-1, 'max':1, 'palette':['brown','white','blue']}

Map.addLayer(dry.select('NDVI'), ndvi_params, 'Dry NDVI')
Map.addLayer(wet.select('NDVI'), ndvi_params, 'Wet NDVI')
Map.addLayer(wet.select('NDWI'), ndwi_params, 'Wet NDWI')
Map

Map(center=[15.178809776080229, 105.11427870578659], controls=(WidgetControl(options=['position', 'transparent…

In [122]:
# โหลดระดับอำเภอ
gaul2 = ee.FeatureCollection("FAO/GAUL/2015/level2")

ubon_districts = gaul2.filter(
    ee.Filter.And(
        ee.Filter.eq("ADM0_NAME", "Thailand"),
        ee.Filter.eq("ADM1_NAME", "Ubon Ratchathani")
    )
)

In [123]:
zonal_ndvi = wet.select('NDVI').reduceRegions(
    collection=ubon_districts,
    reducer=ee.Reducer.mean(),
    scale=10
)

zonal_ndvi.getInfo()

{'type': 'FeatureCollection',
 'columns': {'ADM0_CODE': 'Integer',
  'ADM0_NAME': 'String',
  'ADM1_CODE': 'Integer',
  'ADM1_NAME': 'String',
  'ADM2_CODE': 'Integer',
  'ADM2_NAME': 'String',
  'DISP_AREA': 'String',
  'EXP2_YEAR': 'Integer',
  'STATUS': 'String',
  'STR2_YEAR': 'Integer',
  'Shape_Area': 'Float',
  'Shape_Leng': 'Float',
  'mean': 'Float<-1.0, 1.0>',
  'system:index': 'String'},
 'features': [{'type': 'Feature',
   'geometry': {'type': 'Polygon',
    'coordinates': [[[105.2540437354171, 14.76215727693386],
      [105.2540571294287, 14.76153745369599],
      [105.25410615152026, 14.760957747250643],
      [105.25418642634224, 14.760404859171784],
      [105.25429339996884, 14.759878662381105],
      [105.25442716174466, 14.759361403161027],
      [105.25458324704891, 14.758857536950186],
      [105.25477058532289, 14.758358066231617],
      [105.2549756932913, 14.757867619457436],
      [105.25520312501435, 14.757377060294845],
      [105.25544832658073, 14.756891023

In [124]:
Map.addLayer(
    zonal_ndvi,
    {},
    "District NDVI (Wet)"
)

In [125]:
import pandas as pd

df = pd.DataFrame(
    [f['properties'] for f in zonal_ndvi.getInfo()['features']]
)

df[['ADM2_NAME', 'mean']]

,ADM2_NAME,mean
0,Buntharik,0.663069
1,Det Udom,0.537757
2,Don Mot Daeng,0.565942
3,Khemarat,0.533775
4,Khong Chiam,0.650615
5,Khuang Nai,0.468374
6,Kut Khaopun,0.459975
7,Muang Samsip,0.456962
8,Muang Ubon Ratchathani,0.519696
9,Na Chaluai,0.663437


In [126]:
ndvi_diff = wet.select('NDVI') \
    .subtract(dry.select('NDVI')) \
    .clip(geometry)

vis_params = {
    'min': -0.3,
    'max': 0.3,
    'palette': ['red', 'white', 'green']
}

Map.addLayer(ndvi_diff, vis_params, 'NDVI Change')

district_outline = ubon_districts.style(
    color='black',
    fillColor='00000000',
    width=0.75
)

Map.addLayer(district_outline, {}, 'District Boundaries')

Map.add_colorbar(
    vis_params=vis_params,
    label='NDVI Change (Wet - Dry)'
)

Map.centerObject(geometry, 8)

Map

Map(bottom=30272.0, center=[15.178809776080222, 105.11427870578666], controls=(WidgetControl(options=['positio…

In [127]:
task_image = ee.batch.Export.image.toDrive(
    image=ndvi_diff.visualize(
        min=-0.3,
        max=0.3,
        palette=['red', 'white', 'green']
    ),
    description='Ubon_NDVI_Change',
    folder='GEE_Lab2',
    fileNamePrefix='ubon_ndvi_change',
    region=geometry,
    scale=10,
    crs='EPSG:32648',
    maxPixels=1e13
)

task_image.start()

task_csv = ee.batch.Export.table.toDrive(
    collection=zonal_ndvi,
    description='Ubon_District_NDVI_CSV',
    folder='GEE_Lab2',
    fileNamePrefix='ubon_district_ndvi',
    fileFormat='CSV'
)

task_csv.start()

print(task_image.status()['state'])
print(task_csv.status()['state'])

READY
READY
